# 01 — Data: Tiny Shakespeare + character tokenizer

Step 1 of building a GPT from scratch. In this notebook we:

1. Download the dataset (~1 MB of Shakespeare plays)
2. Look at the text
3. Build a **character-level tokenizer** (each character → an integer)
4. Turn the whole text into a tensor and split it into train / validation
5. See what a training batch looks like — the exact input a GPT learns from

## 1. Download the dataset
Saved to `data/` (git-ignored, so the data never goes into the repo).

In [ ]:
from pathlib import Path
import urllib.request

DATA_DIR = Path("data")  # notebooks run from the project root (see .vscode/settings.json)
DATA_DIR.mkdir(exist_ok=True)
path = DATA_DIR / "tinyshakespeare.txt"

if not path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, path)

text = path.read_text()
print(f"{len(text):,} characters")

## 2. Look at the text

In [ ]:
print(text[:500])

## 3. Character tokenizer

A model can only work with numbers. The simplest tokenizer: list every unique character
in the text, and give each one an id. Our *vocabulary* is just those characters.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print("vocab:", repr("".join(chars)))
print("vocab_size:", vocab_size)

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}  # string -> int
itos = {i: ch for i, ch in enumerate(chars)}  # int -> string


def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]


def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)


print(encode("hello there"))
print(decode(encode("hello there")))

Real LLMs use *subword* tokenizers (e.g. GPT-2's BPE via `tiktoken`) — far fewer tokens
per sentence, but a ~50k vocabulary. Compare:

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
sample = "First Citizen: Before we proceed any further, hear me speak."
print("char tokens:", len(encode(sample)))
print("gpt2 tokens:", len(enc.encode(sample)), enc.encode(sample))

## 4. Encode everything and split train / validation
90% for training, the last 10% to check the model isn't just memorising.

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(data.shape, data.dtype)
print(f"train: {len(train_data):,}  val: {len(val_data):,}")

## 5. What a training batch looks like

A GPT learns to predict the **next token**. We take random chunks of `block_size` tokens
(the context length). The target `y` is the same chunk shifted one position to the right,
so a single chunk holds `block_size` training examples.

In [ ]:
torch.manual_seed(1337)
block_size = 8  # context length
batch_size = 4  # chunks processed in parallel
device = "mps" if torch.backends.mps.is_available() else "cpu"


def get_batch(split: str):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


xb, yb = get_batch("train")
print("x:", xb.shape, "\n", xb)
print("y:", yb.shape, "\n", yb)

In [ ]:
# Every (context -> target) pair inside the first chunk
for t in range(block_size):
    context = xb[0, : t + 1].tolist()
    target = yb[0, t].item()
    print(f"{decode(context)!r:>12} -> {decode([target])!r}")

## Next

We now have `vocab_size`, `encode` / `decode`, and `get_batch`. Step 2 is the simplest
possible model — a **bigram** model that predicts the next character from only the current
one — followed by self-attention to make it a real transformer.